# 2. Removal Rate (연마율) — 알고리즘 교육 자료

CMP APC 의 최종 출력은 "이번 웨이퍼를 **몇 초** 연마할 것인가" 입니다.
연마 시간을 계산하려면 장비·레시피별 **현재 연마율(RR, Å/sec)** 을 알아야 하는데,
RR 은 고정값이 아니라 **패드 등 소모품이 마모될수록 서서히 변하는 값**입니다.

그래서 MICO 는 과거 실적으로부터 "소모품 사용량 → RR" 관계를
**단순선형회귀 `RR = b1 × 소모량 + b0`** 로 학습하고, 상황별로 4종의 계수를 산출합니다.

| 계수 | 회귀 대상 데이터 | 용도 |
|---|---|---|
| `b1, b0` | 전체 기간 | 기본 모델 |
| `b1_weighted, b0_weighted` | 전체 기간 + 최근 구간 가중 | 최근 경향 반영 |
| `b1_current, b0_current` | 현재 패드 사이클(cycle=1)만 | 실시간 보정 |
| `if_b1, if_b0` | 패드 초기 구간(≤ Pad_Seperation)만 | 신품 패드의 상이한 거동 반영 |

이 노트북은 실제 프로덕션 코드 `algorithm_new/Common/REMOVAL_RATE.py` 의 함수들을
**그대로 호출**하면서, 각 계수가 어떤 데이터로 만들어지는지 하나씩 실행하고
트렌드/테이블로 확인합니다.

**사용 방법: 아래 [실행 설정] 셀에서 `FAMILY` 와 `OPER_DESC` 두 값만 원하는 공정으로
바꾸고 위에서부터 순서대로 실행하세요.** RR 학습은 Pre_Thk_VM 학습값(교육 자료 1편)을
입력으로 사용하는데, [준비①] 셀이 자동으로 선행 실행해 줍니다.

```
전체 흐름  (Common/Module.py 의 compute_removal_rate 와 동일한 순서)
────────────────────────────────────────────────────────────────────
 Set-up 조회 (Django DB)  +  merge_df 조회 (MongoDB)
   │
   ├─ [준비①] Pre_Thk_VM 학습 선행 실행  (compute_pre_thk_vm)
   ├─ [준비②] load_pre_thk_data()   _VM 컬럼 결합 (없으면 VM=0)
   │           BIAS 계산              FB_Type=TIME 기준 두께 편차
   │
   ├─ [Step 1] IDLE 필터 + _detect_cycles()   소모품 급감 = 교체 → cycle 번호
   │
   ├─ [Step 2] RR = (투입 전 두께 − CMP 후 두께) / 연마시간,  6σ 이상치 제거
   │
   ├─ [Step 3] Quartile 커버리지 체크   4분위 각 25건 초과해야 회귀 진행
   │
   ├─ [Step 4] _fit_lr()          전체 데이터 회귀        → b1, b0
   ├─ [Step 5] _fit_weighted()    최근 구간 가중 회귀      → b1/b0_weighted
   ├─ [Step 6] _fit_current()     현재 사이클 실시간 회귀   → b1/b0_current
   └─ [Step 7] _fit_if()          패드 초기 구간 회귀      → if_b1, if_b0
────────────────────────────────────────────────────────────────────
```

## 0. 환경 설정

- `algorithm_new/` 를 import 경로에 추가하고, Jupyter(async 이벤트 루프) 위에서
  Django ORM 동기 쿼리를 허용하도록 `DJANGO_ALLOW_ASYNC_UNSAFE` 를 설정합니다.
- 그래프는 기본 matplotlib 스타일 대신, 여백/그리드/색상을 정돈한 커스텀 스타일
  (`style_ax`, `PALETTE`)을 한 번만 정의하고 이후 모든 그래프에 재사용합니다.

In [ ]:
import sys, os
from pathlib import Path

ALGO_DIR = Path('..') / 'algorithm_new'
sys.path.insert(0, str(ALGO_DIR))

# Jupyter 는 async 이벤트 루프 위에서 실행되므로 Django ORM 동기 쿼리 허용
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = '1'

import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)  # findfont 경고 숨김

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

# ── 한글 폰트 자동 탐색 (Mac/Windows/Linux 어디서 열어도 한글이 깨지지 않도록) ──
_KOREAN_FONTS = ['AppleGothic', 'Malgun Gothic', 'NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR']
_installed = {f.name for f in font_manager.fontManager.ttflist}
_font = next((f for f in _KOREAN_FONTS if f in _installed), None)
# 한글 폰트를 1순위로, DejaVu Sans를 2순위로 두면 Å 같은 특수기호는
# 한글 폰트에 없어도 자동으로 DejaVu Sans에서 대체 렌더링됨 (matplotlib 폰트 폴백)
plt.rcParams['font.family'] = [_font, 'DejaVu Sans'] if _font else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ── 깔끔한 트렌드 스타일 (기본 matplotlib 스타일 대신 사용) ──────────────
INK_PRIMARY   = '#0b0b0b'   # 제목/본문
INK_SECONDARY = '#52514e'   # 축 라벨
INK_MUTED     = '#898781'   # 눈금
GRID          = '#e1e0d9'   # 그리드선 (아주 옅게)
BASELINE      = '#c3c2b7'   # 축선
SURFACE       = '#fcfcfb'   # 차트 배경

# 카테고리 색상은 항상 같은 순서로 고정 사용 (장비/채널이 늘어나도 임의로 섞지 않음)
PALETTE = ['#2a78d6', '#1baf7a', '#eda100', '#008300',
           '#4a3aa7', '#e34948', '#e87ba4', '#eb6834']
RAW_COLOR = '#b7d3f6'   # 원본(raw) 산점 — 옅은 블루, 라인과 구분

plt.rcParams.update({
    'figure.facecolor': SURFACE,
    'axes.facecolor'  : SURFACE,
    'axes.edgecolor'  : BASELINE,
    'axes.labelcolor' : INK_SECONDARY,
    'text.color'      : INK_PRIMARY,
    'xtick.color'     : INK_MUTED,
    'ytick.color'     : INK_MUTED,
    'font.size'       : 10,
    'figure.dpi'      : 110,
})

def style_ax(ax, title=None, ylabel=None, xlabel=None, legend=True, legend_loc='best'):
    # 모든 그래프에 공통 적용하는 정돈된 스타일 (spine 정리 + 옅은 그리드 + 좌측 정렬 제목)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    for spine in ('left', 'bottom'):
        ax.spines[spine].set_color(BASELINE)
        ax.spines[spine].set_linewidth(0.8)
    ax.set_axisbelow(True)
    ax.grid(axis='y', color=GRID, linewidth=0.8)
    ax.tick_params(labelsize=8.5, length=0)
    if title:
        ax.set_title(title, fontsize=11, fontweight='bold', color=INK_PRIMARY, loc='left', pad=10)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=9)
    if legend and ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=8, frameon=False, loc=legend_loc)
    return ax

print(f'환경 설정 완료 (한글 폰트: {_font or "시스템 기본값"})')

## ★ 실행 설정 — 여기만 수정하세요

| 변수 | 설명 |
|---|---|
| `FAMILY` | `'DRAM'` 또는 `'NAND'` |
| `OPER_DESC` | 공정 설명 (web Set-up 의 Category.oper_desc 와 동일하게) |
| `KEY_INDEX` | 같은 공정에 실행 키(Lot_Code_Oper_Code_Fab)가 여러 개일 때 선택. 아래 [1. Set-up 조회] 결과 표의 번호 |
| `RR_ROW_INDEX` | 학습 단위(Recipe × APC_Para)가 여러 개일 때 선택. [5. 학습 단위 선택] 결과 표의 번호 |

In [ ]:
FAMILY    = 'DRAM'         # ← 원하는 Family
OPER_DESC = 'M1 CU CMP'    # ← 원하는 공정명

KEY_INDEX    = 0           # 실행 키가 여러 개일 때 선택 (기본: 첫 번째)
RR_ROW_INDEX = 0           # 학습 단위(Recipe × APC_Para)가 여러 개일 때 선택 (기본: 첫 번째)

print(f'실행 대상: Family={FAMILY} | Oper_Desc={OPER_DESC}')

## 1. Set-up 조회 & 실행 키 선택

`Get_data.baseinfoGetData()` 가 web Set-up(Django DB: Category / SubCategory / Detail /
RecipeGroup)에서 해당 공정의 전체 Set-up 정보를 읽어옵니다.

운영 코드(`Common/Module.py` 의 `run()`)와 동일하게
**`Lot_Code + Oper_Code + Fab`** 조합으로 실행 키(`for_key_list`)를 만들고,
키가 여러 개면 `KEY_INDEX` 로 하나를 선택합니다.

In [ ]:
from Common.Get_Data import Get_data
from Common.REMOVAL_RATE import Removal_Rate_Get

# ── Set-up 정보 로드 (Django DB) ─────────────────────────────────────
mico_info_table = Get_data.baseinfoGetData(Family=FAMILY, oper_desc=OPER_DESC)
mico_info_table['Group_Name']   = mico_info_table['Group_Name'].fillna('not_group')
mico_info_table['for_key_list'] = (
    mico_info_table['Lot_Code'] + '_' + mico_info_table['Oper_Code'] + '_' + mico_info_table['Fab']
)

# ── 실행 키 목록 표시 ────────────────────────────────────────────────
key_list = mico_info_table['for_key_list'].unique()
key_summary = pd.DataFrame([{
    '실행 키'   : k,
    'Group'     : mico_info_table.loc[mico_info_table['for_key_list'] == k, 'Group_Name'].iloc[0],
    'Recipe 수' : mico_info_table.loc[mico_info_table['for_key_list'] == k, 'Recipe_ID'].nunique(),
    'Detail 행' : (mico_info_table['for_key_list'] == k).sum(),
} for k in key_list])
print(f'실행 키 {len(key_list)}개 (KEY_INDEX={KEY_INDEX} 선택됨)')
display(key_summary)

# ── KEY_INDEX 로 실행 키 선택 ────────────────────────────────────────
assert 0 <= KEY_INDEX < len(key_list), f'KEY_INDEX 는 0 ~ {len(key_list)-1} 범위로 지정하세요'
FOR_KEY       = key_list[KEY_INDEX]
mico_info_key = mico_info_table[mico_info_table['for_key_list'] == FOR_KEY].copy()

Fab       = mico_info_key['Fab'].unique()[0]
Lot_Code  = mico_info_key['Lot_Code'].unique()[0]
Oper_Code = mico_info_key['Oper_Code'].unique()[0]
Oper_Desc = mico_info_key['Oper_Desc'].unique()[0]
Family    = mico_info_key['Family'].unique()[0]

# pol_type: Category.pol_type (web DB set-up 값) — Module.py _run_pipeline 과 동일
pol_type_vals = mico_info_key['Pol_Type'].dropna().unique()
pol_type = int(pol_type_vals[0]) if len(pol_type_vals) > 0 else None

print(f'\n선택된 키: {FOR_KEY}  (pol_type={pol_type})')
mico_info_key[['Recipe_ID', 'APC_Para', 'Thk_Para', 'Target', 'Pre_Target', 'FB_Type',
               'RR_Para', 'RR_Weight', 'RR_Count', 'RR_Period', 'Pad_Seperation', 'RR_Para_Max']]

## 2. CMP 실측 데이터(merge_df) 조회

`Get_data.MongoDB_GetData()` 로 해당 공정의 CMP 실측 데이터를 조회합니다.
(로컬 테스트 환경에서는 이 함수가 `merge_df_sample.csv` 를 대신 읽도록 되어 있고,
회사 서버에서는 실제 MongoDB 쿼리가 실행됩니다 — 노트북 코드는 동일합니다.)

조회 후 운영 코드(`_run_single`)와 동일하게 `operation_id == Oper_Code` 로 필터링합니다.

In [ ]:
merge_df = Get_data.MongoDB_GetData(Family, Fab, Lot_Code, Oper_Desc)
assert merge_df is not None and not merge_df.empty, '조회된 데이터가 없습니다. Set-up/기간을 확인하세요.'

merge_df['IDLE'] = merge_df['IDLE'].fillna('')
merge_df = merge_df[merge_df['operation_id'] == Oper_Code].copy()
assert not merge_df.empty, f'operation_id == {Oper_Code} 데이터가 없습니다.'

print(f'merge_df : {merge_df.shape[0]:,}행  (기간 {merge_df["Date"].min()} ~ {merge_df["Date"].max()})')
print(f'CMP 장비 : {sorted(merge_df["eqp_id"].unique())}')
print(f'레시피   : {sorted(merge_df["recipe_id"].unique())}')
print(f'IDLE 라벨 분포 (빈 값 = 정상 연속 가동):')
display(merge_df['IDLE'].replace('', '(정상)').value_counts().rename('건수').to_frame().head(10))

## 3. 준비① — Pre_Thk_VM 학습 선행 실행

RR 을 계산하려면 "이 웨이퍼가 **얼마나 두꺼운 상태로 투입되었는가**"(투입 전 두께)를
알아야 합니다. CMP 는 투입 전 두께를 직접 측정하지 않으므로 **교육 자료 1편에서 학습한
Pre_Thk_VM 보정값**을 사용합니다.

운영 환경에서는 파이프라인이 매 실행마다 `Pre_Thk_VM → RR → Offset` 순서로 돌기 때문에
RR 학습 시점에 Pre_Thk_VM 학습값이 항상 MongoDB 에 준비되어 있습니다. 이 노트북에서는
같은 순서를 재현하기 위해 `compute_pre_thk_vm()` 을 먼저 실행합니다.
(로컬 환경에서는 결과가 mock 저장소와 `pre_thk_cache/*.xlsx` 에 저장됩니다.)

In [ ]:
from Common.Module import Module_Get, _MONGO_URL, _MONGO_DB, _build_eqpm_df
from Common.MongoDB_Control import mongodb_controller

# 셀 재실행 대비: 로컬 mock 저장소 초기화
# (회사 실서버의 MongoDB_Control.py 에는 _STORE 가 없으므로 자동으로 건너뜀)
try:
    from Common.MongoDB_Control import _STORE
    for k in list(_STORE):
        _STORE[k] = []
except ImportError:
    _STORE = None   # 회사 실서버 — 실제 MongoDB 에 누적 저장됨

Module_Get.compute_pre_thk_vm(merge_df, mico_info_key, pol_type)

## 4. 준비② — Pre_Thk_VM 로드(`_VM` 컬럼) & 경로 분기

`Common/Module.py` 의 `compute_removal_rate()` 앞부분과 동일한 로직입니다.
Set-up 값에 따라 자동 분기합니다.

| 경로 | 조건 | 처리 |
|---|---|---|
| **VM=0** | `Pre_Oper_Code` 도 `Pre_Oper_Code2` 도 없음 | 투입 전 두께 = `Pre_Target` 고정 |
| **VM=0 + 회귀 보정** | `Pre_Oper_Code2~4` 만 설정 | `apply_pre_oper2_correction()` |
| **Pre_Thk_VM 로드** | `Pre_Oper_Code`(또는 ITM) 설정 | `load_pre_thk_data()` — 채널별 학습값을 `merge_asof` 로 결합 |

`load_pre_thk_data()` 는 준비①에서 저장한 학습값 테이블을 읽어, 각 웨이퍼의
전공정 처리 시각(`pre_oper_time`) 기준 **가장 최근 학습값**을 `pre_eq_ch`(전공정
장비_채널)별로 붙여 `{Thk_Para}_VM` 컬럼을 만듭니다.

In [ ]:
Maker         = mico_info_key['Maker'].unique()[0]
Pre_Oper_Code = mico_info_key['Pre_Oper_Code'].unique()[0]
Thk_Para_List = mico_info_key['Thk_Para'].unique()
Thk_Para_13P  = mico_info_key[mico_info_key['FB_Type'] == 'TIME']['Thk_Para'].unique()[0]

pre_oper2_vals = mico_info_key['Pre_Oper_Code2'].dropna()
has_pre_oper2  = len(pre_oper2_vals[pre_oper2_vals != '']) > 0

# ── Module.py compute_removal_rate 와 동일한 분기 ────────────────────
if Pre_Oper_Code == '' and not has_pre_oper2:
    print('경로: VM=0 (Pre_Oper_Code 미설정) → 투입 전 두께를 Pre_Target 고정값으로 사용')
    merge_df_rr = merge_df.copy()
    for Thk_key in Thk_Para_List:
        merge_df_rr[Thk_key + '_VM'] = 0
else:
    print('경로: Pre_Thk_VM 로드 (load_pre_thk_data)')
    merge_df_rr = Removal_Rate_Get.load_pre_thk_data(merge_df.copy(), mico_info_key, _MONGO_URL, _MONGO_DB)

vm_cols = [c for c in merge_df_rr.columns if c.endswith('_VM')]
print(f'\nmerge_df_rr: {merge_df_rr.shape[0]:,}행  |  생성된 _VM 컬럼: {vm_cols}')
for c in vm_cols:
    print(f'  {c}: 평균 {merge_df_rr[c].mean():.2f} | 결측 {merge_df_rr[c].isna().sum()}건 (결측은 계산 시 0 처리)')

## 5. 학습 단위(Recipe × APC_Para) 선택 & 파라미터 추출

운영 코드는 Set-up 의 모든 행(Recipe × APC_Para 조합)을 루프 돌며 `compute_rr()` 을
호출합니다. 이 노트북은 교육을 위해 `RR_ROW_INDEX` 로 한 행을 골라 그 흐름을 따라갑니다.

- **BIAS** : `Thk_Para − Thk_Para_13P − (Target − Target_13P)` —
  PRESSURE(ED/EX) 파라미터는 CMP 후 두께 대신 TIME 기준 파라미터 대비 편차로 계산합니다.
- **consumable_Para** : `RR_Para` Set-up 값(HEAD/PAD/DISK)에 따라 회귀의 x 축이 될
  소모품 컬럼이 결정됩니다.
- **vm_col / is_rev / is_bias_type** : RR 산식이 Set-up 에 따라 달라지는 부분
  (`_process_models()` 와 동일하게 판단).

In [ ]:
search_key = mico_info_key[mico_info_key['Fab'] == Fab].reset_index(drop=True)
print(f'학습 단위 {len(search_key)}개 (RR_ROW_INDEX={RR_ROW_INDEX} 선택됨)')
display(search_key[['Recipe_ID', 'APC_Para', 'Thk_Para', 'FB_Type', 'RR_Para',
                    'RR_Weight', 'RR_Count', 'RR_Period', 'Pad_Seperation', 'RR_Para_Max']])

assert 0 <= RR_ROW_INDEX < len(search_key), f'RR_ROW_INDEX 는 0 ~ {len(search_key)-1} 범위로 지정하세요'
key = search_key.iloc[RR_ROW_INDEX]

Thk_Para    = key['Thk_Para']
Recipe_ID   = key['Recipe_ID']
APC_Para    = key['APC_Para']
Post_Target = float(key['Target'])
Pre_Target  = float(key['Pre_Target']) if pd.notna(key['Pre_Target']) else None
Target_13P  = float(mico_info_key[(mico_info_key['Recipe_ID'] == Recipe_ID) &
                                  (mico_info_key['FB_Type'] == 'TIME')]['Target'].unique()[0])

# ── BIAS 계산 (Module.py compute_removal_rate 루프와 동일) ──────────
merge_df_rr['BIAS'] = merge_df_rr[Thk_Para] - merge_df_rr[Thk_Para_13P] - (Post_Target - Target_13P)

# ── compute_rr 초입과 동일한 파라미터 결정 ──────────────────────────
Pol_Para  = Get_data.APCParaGet(APC_Para, pol_type)   # 연마 시간 컬럼(들)
Head_Para = Get_data.HeadParaGet(APC_Para)
Pad_Para  = Get_data.PadParaGet(APC_Para)
Disk_Para = Get_data.DiskParaGet(APC_Para)

RR_Para = key['RR_Para'].upper()
if RR_Para == 'HEAD':
    consumable_Para = Head_Para
elif RR_Para == 'PAD':
    consumable_Para = Pad_Para
elif RR_Para == 'DISK':
    consumable_Para = Disk_Para

RR_Weight      = float(key['RR_Weight'])   # 가중 회귀에서 최근 구간 반복 배수
RR_Count       = float(key['RR_Count'])    # current 회귀 최소 데이터 수
RR_Period      = key['RR_Period']          # current 회귀 소모품 구간 폭 (NaN = 사이클 전체)
Pad_Seperation = key['Pad_Seperation']     # IF 회귀 구간 상한 (NaN = 미사용)
RR_Para_Max    = float(key['RR_Para_Max']) if pd.notna(key['RR_Para_Max']) else np.nan

# ── RR 산식 결정 (_process_models 와 동일) ──────────────────────────
Pre_Thk_Para = key['Pre_Thk_Para_ITM']
no_pre_thk   = Pre_Thk_Para == '' or pd.isna(Pre_Thk_Para)
is_bias_type = key['FB_Type'] == 'PRESSURE'
is_rev       = no_pre_thk and 'REV' in Thk_Para
vm_col       = (Thk_Para if no_pre_thk else Pre_Thk_Para) + '_VM'

print(f'선택된 학습 단위: Recipe={Recipe_ID} | APC_Para={APC_Para} | Thk_Para={Thk_Para}')
print(f'  consumable_Para (회귀 x축) : {consumable_Para}  (RR_Para={RR_Para})')
print(f'  Pol_Para (연마 시간)       : {Pol_Para}')
print(f'  vm_col                     : {vm_col}')
print(f'  is_bias_type={is_bias_type} | is_rev={is_rev}')
print(f'  RR_Weight={RR_Weight} | RR_Count={RR_Count} | RR_Period={RR_Period} | Pad_Seperation={Pad_Seperation}')

## Step 1 — IDLE 필터 + `_detect_cycles()` : 패드 사이클 분리

```python
# compute_rr() 내부 — 학습 대상 데이터 필터
temp_data = merge_df[
    (merge_df['operation_id'] == Oper_Code) &
    (merge_df['recipe_id']    == Recipe_ID) &
    ((merge_df['IDLE'] == '') | (merge_df['IDLE'].isna()))   # Idle 직후 웨이퍼 제외
].copy()

# _detect_cycles() 내부 — EQP×Recipe 별로 소모품 사용량이 3 이상 "급감"하면 교체로 판단
raw = (temp_data2[consumable_Para].diff() < -3).cumsum()
temp_data2['cycle'] = raw.max() + 1 - raw     # 최신 패드 = cycle 1, 과거로 갈수록 2, 3, …
```

**왜 IDLE 데이터를 빼는가?** Idle 직후 웨이퍼는 RR 이 일시적으로 변한 상태라
(그 보정이 바로 다음 교육 자료의 Offset), 패드 마모에 따른 정상 RR 경향 학습에
넣으면 회귀가 왜곡됩니다.

In [ ]:
# ── IDLE / Recipe 필터 (compute_rr 내부와 동일) ──────────────────────
temp_data = merge_df_rr[
    (merge_df_rr['operation_id'] == Oper_Code) &
    (merge_df_rr['recipe_id']    == Recipe_ID) &
    ((merge_df_rr['IDLE'] == '') | (merge_df_rr['IDLE'].isna()))
].copy()
print(f'필터 전 {len(merge_df_rr):,}행 → 필터 후 {len(temp_data):,}행 '
      f'(Recipe={Recipe_ID}, IDLE 제외)')

# ── _detect_cycles() 실제 함수 호출 ──────────────────────────────────
temp_data3 = Removal_Rate_Get._detect_cycles(temp_data, consumable_Para)
assert not temp_data3.empty, '사이클 분리 결과가 없습니다.'

print(f'\n_detect_cycles 결과: {len(temp_data3):,}행 | 장비×레시피 조합 {temp_data3["eq_recipe"].nunique()}개')
print('cycle 번호 (1=현재 패드, 클수록 오래된 패드):')
cyc_summary = temp_data3.groupby('cycle').agg(
    건수=('cycle', 'size'),
    소모량_최소=(consumable_Para, 'min'),
    소모량_최대=(consumable_Para, 'max'),
)
display(cyc_summary)

In [ ]:
# ── Step 1 시각화 — 대표 장비의 사이클 분리 (자동 선택: 데이터 최다 조합) ──
DEMO_KEY = temp_data3.groupby('eq_recipe').size().idxmax()
DEMO_EQP, DEMO_RCP = DEMO_KEY.split('//')
demo = temp_data3[temp_data3['eq_recipe'] == DEMO_KEY].sort_values('Date')
cycles = sorted(demo['cycle'].unique())

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle(f'Step 1 — 패드 사이클 분리  [{DEMO_EQP} / {DEMO_RCP}]  (자동 선택)',
             fontsize=12, fontweight='bold', color=INK_PRIMARY, x=0.01, ha='left')

# ① 소모품 사용량 트렌드 (급감 = 교체 시점)
ax = axes[0]
for cyc in cycles:
    c = demo[demo['cycle'] == cyc]
    ax.scatter(c['Date'], c[consumable_Para], s=6, alpha=0.6,
               color=PALETTE[(int(cyc) - 1) % len(PALETTE)], linewidths=0, label=f'cycle {cyc}')
style_ax(ax, title=f'{consumable_Para} — 소모품 사용량 (급감 = 교체)', ylabel='사용량')

# ② 전체 장비 소모량 트렌드 (배경 비교)
ax = axes[1]
for i, er in enumerate(sorted(temp_data3['eq_recipe'].unique())):
    sub = temp_data3[temp_data3['eq_recipe'] == er].sort_values('Date')
    ax.scatter(sub['Date'], sub[consumable_Para], s=4, alpha=0.35,
               color=PALETTE[i % len(PALETTE)], linewidths=0, label=er.split('//')[0])
style_ax(ax, title='전체 장비 소모품 트렌드', ylabel='사용량', xlabel='시간', legend_loc='upper right')
axes[-1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## Step 2 — RR 계산 & 6σ 이상치 제거

`_process_models()` 는 eqp_model(설비 기종)별로 루프를 돌며 아래를 수행합니다.
이 노트북은 데이터가 가장 많은 기종 하나를 따라갑니다.

```python
# _process_models() 내부
pre_thk  = Pre_Target + temp_data4[vm_col]                    # 투입 전 두께 = 목표 + VM 보정
post_thk = (Post_Target + BIAS)  if is_bias_type else  temp_data4[Thk_Para]

temp_data4['RR'] = (pre_thk - post_thk) / temp_data4['Pol_Time']   # Å/sec
#                   (is_rev 이면 post − pre : REV 파라미터는 두께가 "쌓이는" 방향)

# 6σ 이상치 제거 — 명백한 계측 오류만 걸러냄
temp_data4 = temp_data4[|RR − mean| < 6σ]
```

**왜 6σ 인가?** RR 자체가 패드 마모에 따라 변하는 값이므로 좁은 기준(2~3σ)으로
자르면 정상적인 마모 경향까지 지워집니다. 6σ 는 계측 실패 수준의 극단값만 제거합니다.

In [ ]:
# ── eqp_model 선택 (운영 코드는 모델별 루프 — 여기선 데이터 최다 모델) ──
model_sizes = temp_data3.groupby('eqp_model').size()
x = model_sizes.idxmax()
print(f'eqp_model {len(model_sizes)}종 중 선택: {x} ({model_sizes[x]:,}행)')

temp_data4 = temp_data3[temp_data3['eqp_model'] == x].copy()

# ── 소모품 범위 & 4분위 기준점 (RR_Para_Max 설정 시 상한 고정) ───────
part_max = temp_data4[consumable_Para].max()
part_min = temp_data4[consumable_Para].min()
if pd.notna(RR_Para_Max):
    part_max = RR_Para_Max
part_1q = part_min + (part_max - part_min) / 4
part_2q = part_min + (part_max - part_min) / 2
part_3q = part_min + (part_max - part_min) * 3 / 4

# ── 컬럼 선택 → Pol_Time → RR 계산 (_process_models 와 동일) ─────────
col_list = ['Date', 'substrate_id', 'eqp_id', 'recipe_id', 'process_id',
            Pad_Para, Head_Para, Disk_Para, 'pre_eq_ch',
            vm_col, Thk_Para, 'cycle', 'BIAS'] + Pol_Para
temp_data4 = temp_data4[[c for c in dict.fromkeys(col_list) if c in temp_data4.columns]].copy()
temp_data4.drop_duplicates(inplace=True)

temp_data4['Pol_Time'] = temp_data4[Pol_Para].sum(axis=1)
temp_data4.dropna(axis=0, subset=[Thk_Para, consumable_Para], inplace=True)
temp_data4.drop(temp_data4[temp_data4['Pol_Time'] == 0].index, inplace=True)
temp_data4.fillna(value={vm_col: 0}, inplace=True)

vm       = temp_data4[vm_col]
pre_thk  = Pre_Target + vm
post_thk = (Post_Target + temp_data4['BIAS']) if is_bias_type else temp_data4[Thk_Para]
temp_data4['RR'] = ((post_thk - pre_thk) if is_rev else (pre_thk - post_thk)) / temp_data4['Pol_Time']

temp_data4['eq_recipe'] = temp_data4['eqp_id'] + '//' + temp_data4['recipe_id']

# ── 6σ 이상치 제거 ───────────────────────────────────────────────────
rr_avg, rr_std = temp_data4['RR'].mean(), temp_data4['RR'].std()
before_n = len(temp_data4)
temp_data4 = temp_data4[(temp_data4['RR'] < rr_avg + rr_std * 6)
                       & (temp_data4['RR'] > rr_avg - rr_std * 6)].copy()

print(f'RR 평균 {rr_avg:.2f} Å/sec | 표준편차 {rr_std:.2f}')
print(f'6σ 이상치 제거: {before_n:,}행 → {len(temp_data4):,}행 ({before_n - len(temp_data4)}건 제거)')
temp_data4[['Date', 'eqp_id', 'recipe_id', consumable_Para, vm_col, Thk_Para, 'Pol_Time', 'RR']].head(5)

In [ ]:
# ── Step 2 시각화 — RR 분포와 소모품 vs RR 관계 ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Step 2 — RR = (투입 전 두께 − CMP 후 두께) / 연마시간',
             fontsize=12, fontweight='bold', color=INK_PRIMARY, x=0.01, ha='left')

# ① 장비별 RR 분포
ax = axes[0]
for i, eqp in enumerate(sorted(temp_data4['eqp_id'].unique())):
    sub = temp_data4[temp_data4['eqp_id'] == eqp]
    ax.hist(sub['RR'], bins=40, alpha=0.5, color=PALETTE[i % len(PALETTE)], label=eqp)
ax.axvline(rr_avg, color=INK_PRIMARY, linewidth=1.2, linestyle='--', label=f'평균 {rr_avg:.1f}')
style_ax(ax, title='장비별 RR 분포', xlabel='RR (Å/sec)', ylabel='건수')

# ② 대표 장비: 소모품 사용량 vs RR (사이클별 색)
ax = axes[1]
demo4 = temp_data4[temp_data4['eq_recipe'] == DEMO_KEY]
for cyc in sorted(demo4['cycle'].unique()):
    c = demo4[demo4['cycle'] == cyc]
    ax.scatter(c[consumable_Para], c['RR'], s=8, alpha=0.5,
               color=PALETTE[(int(cyc) - 1) % len(PALETTE)], linewidths=0, label=f'cycle {cyc}')
style_ax(ax, title=f'{DEMO_EQP} — {consumable_Para} vs RR', xlabel=f'{consumable_Para} (사용량)', ylabel='RR (Å/sec)')

plt.tight_layout()
plt.show()
print('→ 소모품 사용량에 따라 RR 이 선형적으로 변하는 경향이 보이면 회귀 학습이 유효합니다.')

## Step 3 — Quartile 커버리지 체크

장비×레시피(`eq_recipe`)별로 회귀를 수행하기 전에, 소모품 사용량 범위를 4등분(Q1~Q4)해
**각 구간에 데이터가 25건을 초과**하는지 확인합니다. 데이터가 특정 구간(예: 신품 구간)에만
몰려 있으면 기울기(b1)가 불안정해지기 때문입니다.

```python
# _process_models() 내부
if Count < 10: continue                          # 조합 전체 10건 미만이면 스킵

quartile_counts = pd.cut(temp_data5[consumable_Para],
                         bins=[part_min, part_1q, part_2q, part_3q, part_max],
                         labels=['Q1', 'Q2', 'Q3', 'Q4']).value_counts()

if (quartile_counts > 25).all():                 # 4분위 모두 25건 초과 → 회귀 진행
    b1, b0 = _fit_lr(...)
```

In [ ]:
def quartile_check(df):
    qc = pd.cut(df[consumable_Para],
                bins=[part_min, part_1q, part_2q, part_3q, part_max],
                labels=['Q1', 'Q2', 'Q3', 'Q4']).value_counts()
    return qc, bool((qc > 25).all())

# ── 대표 학습 조합 선택: 데이터 최다 & quartile 통과 조합 우선 ────────
order = temp_data4.groupby('eq_recipe').size().sort_values(ascending=False).index.tolist()
DEMO_KEY = next((er for er in order if quartile_check(temp_data4[temp_data4['eq_recipe'] == er])[1]),
                order[0])
DEMO_EQP, DEMO_RCP = DEMO_KEY.split('//')
temp_data5 = temp_data4[temp_data4['eq_recipe'] == DEMO_KEY].copy()

quartile_counts, q_ok = quartile_check(temp_data5)
print(f'대표 조합: {DEMO_EQP} / {DEMO_RCP}  ({len(temp_data5):,}건)')
print(f'소모품 4분위 경계: {part_min:.1f} | {part_1q:.1f} | {part_2q:.1f} | {part_3q:.1f} | {part_max:.1f}')
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    n = int(quartile_counts.get(q, 0))
    print(f'  {q}: {n:5,}건  {"통과" if n > 25 else "미달"}')
print(f'\n→ 회귀 진행 가능: {q_ok}' + ('' if q_ok else '  (운영 코드는 이 조합을 스킵합니다 — 교육을 위해 계속 진행)'))

In [ ]:
# ── Step 3 시각화 — 전체 장비 quartile 커버리지 ──────────────────────
eq_list = order
qc_df = pd.DataFrame({er.split('//')[0]: quartile_check(temp_data4[temp_data4['eq_recipe'] == er])[0]
                      for er in eq_list}).T[['Q1', 'Q2', 'Q3', 'Q4']]

fig, ax = plt.subplots(figsize=(max(9, 1.5 * len(qc_df)), 4.5))
xpos = np.arange(len(qc_df.index))
width = 0.2
for j, q in enumerate(['Q1', 'Q2', 'Q3', 'Q4']):
    ax.bar(xpos + (j - 1.5) * width, qc_df[q], width, label=q,
           color=PALETTE[j], alpha=0.85)
ax.axhline(25, color=PALETTE[5], linewidth=1.3, linestyle='--', label='기준 25건')
ax.set_xticks(xpos)
ax.set_xticklabels(qc_df.index, rotation=20)
style_ax(ax, title='Step 3 — 장비별 Quartile 커버리지 (모든 구간 25건 초과 시 회귀 진행)', ylabel='건수')
plt.tight_layout()
plt.show()

## Step 4 — `_fit_lr()` : b1, b0 (기본 선형회귀)

```python
# REMOVAL_RATE.py  _fit_lr()
lr.fit(df[consumable_Para], df['RR'])
b1 = round(lr.coef_[0][0],   4)    # 기울기: 소모품 1 단위 증가 시 RR 변화량
b0 = round(lr.intercept_[0], 4)    # 절편  : 소모품 = 0 (신품) 기준 RR
```

APC 는 이 계수로 임의 시점의 RR 을 예측합니다: `RR_예측 = b1 × 현재 소모량 + b0`.
일반적으로 패드가 마모될수록 RR 이 감소하므로 `b1 < 0` 이 자연스러운 값입니다.

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
b1, b0 = Removal_Rate_Get._fit_lr(temp_data5, consumable_Para, 'RR', lr)

print(f'b1 = {b1}   (소모품 1 단위 증가 시 RR 변화량)')
print(f'b0 = {b0}   (신품(소모=0) 기준 RR)')
print(f'\n해석: {consumable_Para} 가 1 증가할 때마다 RR 이 {b1:+.4f} Å/sec 변화,')
print(f'      신품 기준 RR = {b0:.2f} Å/sec')

# ── 시각화: 회귀선 + 전체 장비 b1/b0 비교 ────────────────────────────
b1_all = {}
for er in eq_list:
    td = temp_data4[temp_data4['eq_recipe'] == er]
    if len(td) >= 10 and quartile_check(td)[1]:
        b1_all[er.split('//')[0]] = Removal_Rate_Get._fit_lr(td, consumable_Para, 'RR', lr)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle(f'Step 4 — _fit_lr()  b1={b1}, b0={b0}',
             fontsize=12, fontweight='bold', color=INK_PRIMARY, x=0.01, ha='left')

ax = axes[0]
ax.scatter(temp_data5[consumable_Para], temp_data5['RR'], s=6, alpha=0.35,
           color=RAW_COLOR, linewidths=0, label='실측 RR')
x_rng = np.linspace(temp_data5[consumable_Para].min(), temp_data5[consumable_Para].max(), 100)
ax.plot(x_rng, b1 * x_rng + b0, color=PALETTE[0], linewidth=2,
        label=f'RR = {b1}×x + {b0}')
style_ax(ax, title=f'{DEMO_EQP} — 전체 데이터 회귀', xlabel=consumable_Para, ylabel='RR (Å/sec)')

ax = axes[1]
if b1_all:
    names = list(b1_all.keys())
    vals1 = [b1_all[n][0] for n in names]
    bars = ax.bar(names, vals1, color=PALETTE[0], width=0.55)
    ax.bar_label(bars, fmt='%.4f', fontsize=8, padding=3, color=INK_SECONDARY)
    ax.axhline(0, color=BASELINE, linewidth=1.2)
    style_ax(ax, title='장비별 b1 비교 (quartile 통과 조합만)', ylabel='b1', legend=False)
    ax.tick_params(axis='x', rotation=20)
else:
    ax.text(0.5, 0.5, 'quartile 통과 조합 없음', ha='center', va='center')

plt.tight_layout()
plt.show()

## Step 5 — `_fit_weighted()` : b1_weighted, b0_weighted (최근 가중 회귀)

전체 기간 회귀(b1/b0)는 오래된 데이터와 최근 데이터를 똑같이 취급합니다.
슬러리 교체 등으로 **최근 RR 수준이 이동**했다면 최근 데이터를 더 믿어야 하므로,
기간을 5구간으로 나눠 **최근 구간을 `RR_Weight` 배 반복 복제**한 뒤 회귀합니다.

```python
# _process_models() 내부 — 세그먼트 구성
group    = pd.cut(Date, bins=5, labels=False)          # 시간 5구간 (0=과거, 4=최근)
part_seg = pd.cut(consumable, bins=4)                  # 소모량 4구간

# 최근(4_*) 세그먼트가 4개 part 모두 25건 초과일 때만 산출 (include_recent)
include_recent = (최근 구간의 part_seg 4종 모두 존재) and (각 25건 초과)

# _fit_weighted() 내부
weight = [1, 2, 3, 4, RR_Weight]                       # group 0→1배 … 4→RR_Weight배
X_w    = np.repeat(consumable, weight[group])          # 행 복제 = 가중치
y_w    = np.repeat(RR,         weight[group])
```

**include_recent 조건의 의미**: 최근 구간에 소모량 전 범위의 데이터가 고루 없으면
가중 회귀가 오히려 왜곡되므로 산출하지 않고 `'-'` 를 반환합니다.

In [ ]:
# ── date × part 세그먼트 구성 (_process_models 와 동일) ──────────────
temp_data5['group']         = pd.cut(temp_data5['Date'], bins=5, labels=False)
temp_data5['date_seg']      = pd.cut(temp_data5['Date'], bins=5, labels=[0, 1, 2, 3, 4]).astype(str)
temp_data5['part_seg']      = pd.cut(temp_data5[consumable_Para], bins=4, labels=[0, 1, 2, 3]).astype(str)
temp_data5['date_part_seg'] = temp_data5['date_seg'] + '_' + temp_data5['part_seg']

seg_counts     = temp_data5.groupby(['eqp_id', 'date_part_seg']).size().reset_index(name='count')
recent_counts  = seg_counts[seg_counts['date_part_seg'].str.startswith('4_')]
include_recent = (len(recent_counts['date_part_seg'].unique()) == 4) and (recent_counts['count'] > 25).all()

print(f'include_recent = {include_recent}')
print('최근 구간(date_seg=4)의 part 별 건수:')
print(recent_counts[['date_part_seg', 'count']].to_string(index=False))

# ── _fit_weighted() 실제 함수 호출 ───────────────────────────────────
weight = [1, 2, 3, 4, RR_Weight]
weighted_b1, weighted_b0 = Removal_Rate_Get._fit_weighted(temp_data5, consumable_Para, weight, include_recent, lr)

print(f'\nb1_weighted = {weighted_b1}   (기본 b1 = {b1})')
print(f'b0_weighted = {weighted_b0}   (기본 b0 = {b0})')
if weighted_b1 == '-':
    print('→ include_recent=False → 가중 회귀 미산출 ("-")')

In [ ]:
# ── Step 5 시각화 — 세그먼트 heatmap + 회귀선 비교 ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
fig.suptitle('Step 5 — date × part 세그먼트 & 가중 회귀',
             fontsize=12, fontweight='bold', color=INK_PRIMARY, x=0.01, ha='left')

# ① date_seg × part_seg 건수 heatmap
ax = axes[0]
pivot = temp_data5.groupby(['date_seg', 'part_seg']).size().unstack(fill_value=0)
pivot = pivot.reindex(index=['0', '1', '2', '3', '4'], columns=['0', '1', '2', '3'], fill_value=0)
im = ax.imshow(pivot.values, aspect='auto', cmap='Blues', vmin=0)
plt.colorbar(im, ax=ax, label='건수')
ax.set_xticks(range(4)); ax.set_xticklabels(['part 0\n(신품측)', 'part 1', 'part 2', 'part 3\n(마모측)'], fontsize=8)
ax.set_yticks(range(5)); ax.set_yticklabels(['date 0\n(과거)', 'date 1', 'date 2', 'date 3', 'date 4\n(최근)'], fontsize=8)
for i in range(5):
    for j in range(4):
        n = pivot.values[i, j]
        mark = ' ○' if (i == 4 and n > 25) else (' ×' if i == 4 else '')
        ax.text(j, i, f'{n}{mark}', ha='center', va='center', fontsize=8,
                color='white' if n > pivot.values.max() * 0.6 else INK_PRIMARY)
ax.set_title('date×part 건수 (마지막 행이 include_recent 조건)', fontsize=10, loc='left', pad=8)

# ② 기본 vs 가중 회귀선
ax = axes[1]
ax.scatter(temp_data5[consumable_Para], temp_data5['RR'], s=5, alpha=0.25,
           color=RAW_COLOR, linewidths=0)
recent_pts = temp_data5[temp_data5['group'] == 4]
ax.scatter(recent_pts[consumable_Para], recent_pts['RR'], s=9, alpha=0.6,
           color=PALETTE[2], linewidths=0, label=f'최근 구간 (×{RR_Weight:.0f} 가중)')
ax.plot(x_rng, b1 * x_rng + b0, color=PALETTE[0], linewidth=2, label=f'기본 b1={b1}')
if weighted_b1 != '-':
    ax.plot(x_rng, weighted_b1 * x_rng + weighted_b0, color=PALETTE[3], linewidth=2,
            linestyle='--', label=f'가중 b1={weighted_b1}')
style_ax(ax, title=f'{DEMO_EQP} — 기본 vs 가중 회귀선', xlabel=consumable_Para, ylabel='RR (Å/sec)')

plt.tight_layout()
plt.show()

## Step 6 — `_get_current_cycle()` + `_fit_current()` : b1_current, b0_current

**현재 패드(cycle=1)** 데이터만으로 실시간 회귀를 수행합니다. 패드 개체차를 가장 잘
반영하지만 데이터가 적어 위험하므로, **세 가지 조건을 모두 만족**할 때만 산출하고
계수 범위도 기본/가중 회귀 기준으로 제한(clamp)합니다.

```python
# _fit_current() 내부 — 산출 조건
count      > RR_Count                    # ① 현재 사이클 데이터 수 충분
now − 최신 타점 < 12시간                  # ② 데이터가 살아있는 상태 (12h 이내 생산)
PM 이후 웨이퍼 수(rank) ≥ 2               # ③ PM 직후 불안정 구간 제외

# 계수 clamp (RR_Period 미설정 시)
current_b1 : 기준 b1 의 0 ~ 2배 범위로 제한
current_b0 : 기준 b0 의 ±10% 범위로 제한
```

조건 ③의 PM rank 는 `_build_eqpm_df()` 가 MES 설비 이벤트 로그(PM/EndLot)로부터
계산합니다. **로컬 샘플 데이터는 과거 데이터라 조건 ②를 충족하지 못해 보통
`'-'`(미산출)가 됩니다** — 회사 서버에서 실시간 데이터로 실행하면 산출됩니다.

In [ ]:
from datetime import datetime, timedelta

# ── _get_current_cycle() 실제 함수 호출 ──────────────────────────────
current_cycle = Removal_Rate_Get._get_current_cycle(temp_data5, consumable_Para, RR_Period)
print(f'현재 사이클(cycle=1) 데이터: {len(current_cycle):,}건'
      + ('' if pd.isna(RR_Period) else f'  (RR_Period={RR_Period} 구간 한정)'))

# ── EQPM_df 구성 (_build_eqpm_df 실제 함수 — PM 이후 누적 웨이퍼 rank) ──
EQPM_df = _build_eqpm_df(merge_df_rr, Maker, Fab)
pm_rank = Removal_Rate_Get._get_pm_rank(EQPM_df, Maker, DEMO_EQP, DEMO_RCP)

# ── 조건 확인 ────────────────────────────────────────────────────────
if not current_cycle.empty:
    latest = current_cycle['Date'].iloc[-1]
    elapsed = datetime.now() - latest
    print(f'조건① 데이터 수      : {len(current_cycle)} > {RR_Count:.0f} → {len(current_cycle) > RR_Count}')
    print(f'조건② 최신 타점 경과 : {elapsed} < 12h → {elapsed < timedelta(hours=12)}')
    print(f'조건③ PM rank        : {pm_rank} >= 2 → {pm_rank >= 2}')

# ── _fit_current() 실제 함수 호출 ────────────────────────────────────
Simul_Date, current_b1, current_b0 = Removal_Rate_Get._fit_current(
    current_cycle, consumable_Para, datetime.now(),
    RR_Count, EQPM_df, Maker, DEMO_EQP, DEMO_RCP,
    weighted_b1, weighted_b0, b1, b0, RR_Period, lr
)
print(f'\nb1_current = {current_b1} | b0_current = {current_b0} | Simul_Date = {Simul_Date}')
if current_b1 == '-':
    print('→ 조건 미충족 → current 미산출 ("-") — 로컬 샘플(과거 데이터)에서는 정상 동작입니다')

# ── 시각화: 현재 사이클 데이터 위치 ──────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))
prev = temp_data5[temp_data5['cycle'] != 1]
ax.scatter(prev['Date'], prev['RR'], s=5, alpha=0.25, color=RAW_COLOR,
           linewidths=0, label='이전 패드 (cycle 2+)')
if not current_cycle.empty:
    ax.scatter(current_cycle['Date'], current_cycle['RR'], s=12, alpha=0.7,
               color=PALETTE[5], linewidths=0, label=f'현재 패드 (cycle=1, {len(current_cycle)}건)')
style_ax(ax, title=f'Step 6 — {DEMO_EQP} 현재 사이클 데이터', ylabel='RR (Å/sec)', xlabel='시간')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## Step 7 — `_fit_if()` : if_b1, if_b0 (패드 초기 구간 회귀)

신품 패드는 표면이 아직 길들지 않아 초기 구간(`소모량 ≤ Pad_Seperation`)의 RR 거동이
이후와 다를 수 있습니다. 그래서 초기 구간만 별도로 회귀한 IF 모델을 추가 산출합니다.

```python
# _fit_if() 내부
temp_data7 = temp_data5[consumable ≤ Pad_Seperation]     # 초기 구간만

if len(temp_data7) < 100: return '-', '-'                # 최소 100건
if (초기 구간 4분위 각각 < 25건).any(): return '-', '-'    # 구간 내 고른 분포 필요

return _fit_lr(temp_data7, ...)                          # 초기 구간 회귀
```

`Pad_Seperation` 이 Set-up 에 없으면(NaN) 이 계수는 산출하지 않습니다.

In [ ]:
# ── _fit_if() 실제 함수 호출 ─────────────────────────────────────────
if_b1, if_b0 = Removal_Rate_Get._fit_if(temp_data5, consumable_Para, Pad_Seperation, lr)
print(f'if_b1 = {if_b1} | if_b0 = {if_b0}')

if pd.isna(Pad_Seperation):
    print('→ Pad_Seperation 미설정 — 이 공정은 IF 회귀 대상이 아닙니다 (건너뜀)')
else:
    sep   = float(Pad_Seperation)
    td_if = temp_data5[temp_data5[consumable_Para] <= sep]
    step  = sep / 4
    if_qc = pd.cut(td_if[consumable_Para], bins=[0, step, step * 2, step * 3, sep],
                   labels=['Q1', 'Q2', 'Q3', 'Q4']).value_counts()
    print(f'IF 구간({consumable_Para} ≤ {sep:.0f}) 데이터: {len(td_if)}건 (최소 100건)')
    for q in ['Q1', 'Q2', 'Q3', 'Q4']:
        n = int(if_qc.get(q, 0))
        print(f'  {q}: {n:4,}건  {"통과" if n >= 25 else "미달"}')

    # ── 시각화: IF 구간 회귀선 vs 기본 회귀선 ────────────────────────
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.scatter(temp_data5[consumable_Para], temp_data5['RR'], s=6, alpha=0.3,
               color=RAW_COLOR, linewidths=0, label='전체 데이터')
    ax.scatter(td_if[consumable_Para], td_if['RR'], s=10, alpha=0.6,
               color=PALETTE[2], linewidths=0, label=f'IF 구간 (≤ {sep:.0f})')
    ax.axvline(sep, color=PALETTE[5], linewidth=1.3, linestyle='--', label=f'Pad_Seperation={sep:.0f}')
    ax.plot(x_rng, b1 * x_rng + b0, color=PALETTE[0], linewidth=2, label=f'기본 b1={b1}')
    if if_b1 != '-':
        x_if = np.linspace(0, sep, 50)
        ax.plot(x_if, float(if_b1) * x_if + float(if_b0), color=PALETTE[3], linewidth=2,
                linestyle='-.', label=f'IF if_b1={if_b1}')
    style_ax(ax, title=f'Step 7 — {DEMO_EQP} IF 구간 회귀', xlabel=consumable_Para, ylabel='RR (Å/sec)')
    plt.tight_layout()
    plt.show()

## 최종 — 운영 함수 전체 실행 & 학습값 확인

지금까지 대표 조합 1개를 따라왔습니다. 실제 운영은 `Module_Get.compute_removal_rate()`
가 모든 학습 단위(Recipe × APC_Para) × 모든 장비 조합에 대해 위 단계를 반복하고,
결과를 MongoDB 컬렉션 **`MICO_Removal_Rate_{Lot_Code}_{Oper_Desc}_{Fab}`** 에 저장합니다.
(조건 미충족으로 `'-'` 인 계수는 저장 시 항목에서 제외됩니다.)

이 학습값은 **Offset 학습**(다음 교육 자료)과 **APC 연마 시간 계산**에서
`RR_예측 = B1 × 현재 소모량 + B0` 로 사용됩니다.

In [ ]:
# ── 운영 함수 전체 실행 & MongoDB 에서 학습값 조회 ───────────────────
# 로컬 mock / 회사 실서버 공통 — 저장소 접근은 mongodb_controller 인터페이스만 사용
from datetime import datetime

# 셀 재실행 대비: 로컬 mock 인 경우에만 이전 RR 결과 초기화 (회사 MongoDB 는 건드리지 않음)
if _STORE is not None:
    for k in [k for k in _STORE if 'Removal_Rate' in k]:
        _STORE[k] = []

run_start = datetime.now()
Module_Get.compute_removal_rate(merge_df.copy(), mico_info_key, pol_type)

# 운영 코드(OFFSET_Get.load_rr_data)와 동일한 방식으로 컬렉션 전체를 조회
rr_collection = 'MICO_Removal_Rate_' + Lot_Code + '_' + Oper_Desc + '_' + Fab
rr_result = mongodb_controller(_MONGO_URL, _MONGO_DB, rr_collection).get_df()
if '_id' in rr_result.columns:          # 실서버 pymongo 조회 시 ObjectId 컬럼 제거
    rr_result = rr_result.drop(columns='_id')
assert not rr_result.empty, 'RR 학습값이 생성되지 않았습니다. 데이터/Set-up 조건을 확인하세요.'

# 회사 서버 컬렉션에는 과거 실행 결과가 누적되므로 이번 실행분만 필터링
if 'Date' in rr_result.columns:
    rr_result['Date'] = pd.to_datetime(rr_result['Date'])
    rr_result = rr_result[rr_result['Date'] >= run_start].reset_index(drop=True)
assert not rr_result.empty, '이번 실행에서 생성된 RR 학습값이 없습니다. 데이터/Set-up 조건을 확인하세요.'

num_cols = [c for c in ['b1', 'b0', 'b1_weighted', 'b0_weighted', 'if_b1', 'if_b0'] if c in rr_result.columns]
rr_result[num_cols] = rr_result[num_cols].apply(pd.to_numeric, errors='coerce')

print(f'\n최종 RR 학습값: {len(rr_result)}건 (컬렉션: {rr_collection})')
show_cols = [c for c in ['APC_Para', 'EQ', 'Recipe_ID', 'Count', 'b1', 'b0',
                         'b1_weighted', 'b0_weighted', 'b1_current', 'b0_current',
                         'if_b1', 'if_b0'] if c in rr_result.columns]
display(rr_result[show_cols])

In [ ]:
# ── 최종 시각화 — 선택 APC_Para 의 장비별 계수 비교 ──────────────────
plot_df = rr_result[rr_result['APC_Para'] == APC_Para].copy()
if plot_df.empty:
    plot_df = rr_result.copy()

eq_labels = plot_df['EQ'].tolist()
xpos = np.arange(len(eq_labels))
width = 0.28

fig, axes = plt.subplots(2, 1, figsize=(max(10, 1.6 * len(eq_labels)), 8))
fig.suptitle(f'최종 RR 학습값 — {FOR_KEY} / APC_Para={APC_Para}',
             fontsize=12, fontweight='bold', color=INK_PRIMARY, x=0.01, ha='left')

coef_sets = [('b1', 'b0', '기본', PALETTE[0]),
             ('b1_weighted', 'b0_weighted', '가중', PALETTE[3]),
             ('if_b1', 'if_b0', 'IF', PALETTE[2])]

for ax, idx in zip(axes, [0, 1]):
    j = 0
    for c1, c0, name, color in coef_sets:
        col = (c1 if idx == 0 else c0)
        if col in plot_df.columns and plot_df[col].notna().any():
            bars = ax.bar(xpos + (j - 1) * width, plot_df[col], width, label=name, color=color, alpha=0.85)
            ax.bar_label(bars, fmt='%.3f', fontsize=7, padding=2, color=INK_SECONDARY, rotation=90)
            j += 1
    ax.axhline(0, color=BASELINE, linewidth=1.2)
    ax.set_xticks(xpos)
    ax.set_xticklabels(eq_labels, rotation=20)
    style_ax(ax, title=('기울기 계열 (b1) — 소모품 1 단위당 RR 변화' if idx == 0
                        else '절편 계열 (b0) — 신품 기준 RR'), ylabel=('b1' if idx == 0 else 'b0'))

plt.tight_layout()
plt.show()

## 정리

| Step | 함수 | 입력 → 출력 |
|---|---|---|
| 준비① | `Module_Get.compute_pre_thk_vm()` | 교육 자료 1편 — Pre_Thk_VM 학습값 생성 |
| 준비② | `load_pre_thk_data()` | 학습값 → `_VM` 컬럼 결합 (미설정 시 VM=0) |
| 1 | `_detect_cycles()` | 소모품 급감 → 패드 cycle 번호 (1=현재) |
| 2 | (`_process_models()` 내부) | 두께·연마시간 → RR (Å/sec), 6σ 제거 |
| 3 | (quartile 체크) | 4분위 각 25건 초과 시에만 회귀 진행 |
| 4 | `_fit_lr()` | 전체 데이터 → b1, b0 |
| 5 | `_fit_weighted()` | 최근 가중 → b1/b0_weighted (include_recent 시) |
| 6 | `_fit_current()` | 현재 사이클 → b1/b0_current (3조건 충족 시) |
| 7 | `_fit_if()` | 패드 초기 구간 → if_b1, if_b0 (Pad_Seperation 설정 시) |

실제 운영에서는 `Common/Module.py` 의 `Module_Get.compute_removal_rate()` 가
위 단계를 모든 학습 단위에 대해 실행한 뒤 MongoDB 에 저장하고, 이 학습값은
**Offset 학습**(다음 교육 자료)에서 "패드 모델이 예측하는 정상 RR" 로 사용됩니다.

> 다른 공정을 보려면 맨 위 [실행 설정] 셀에서 `FAMILY` / `OPER_DESC`
> (필요시 `KEY_INDEX` / `RR_ROW_INDEX`)만 바꾸고 전체 재실행하면 됩니다.